# 00 — UC2 Data Analysis & Partition Generation

**Use Case 2**: Wi-Fi AP Load Prediction (FLAG dataset)

This notebook:
1. Loads and explores the raw FLAG dataset
2. Visualizes AP load distributions
3. Generates Dirichlet-partitioned datasets for α ∈ {0.5, 1.0, 5.0, 10.0}

In [ ]:
import sys, os
sys.path.append("..")
import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import Counter


sys.path.insert(0, uc2.LIB_DIR)

## 1. Load Raw Dataset

**IMPORTANT**: Place the pickle file `pickle_2019-05-13-on7_2min.pkl` in `data/raw/`.
Get this file from Francesc.

In [3]:
raw_path = uc2.get_raw_dataset_path()
print(f"Loading dataset from: {raw_path}")

sys.path.insert(0, os.path.abspath(".."))
from data.FlagsRegression.dataset import load_dataset
raw_data = load_dataset(path=raw_path, cache=False)

print(f"\nDataset shape: {raw_data.shape}")
print(f"Columns: {list(raw_data.columns)}")
print(f"Date range: {raw_data['datetime'].min()} → {raw_data['datetime'].max()}")
print(f"Number of unique APs: {raw_data['AP ID'].nunique()}")
print(f"\nFirst rows:")
raw_data.head()

FileNotFoundError: Raw dataset not found at: /mnt/c/Users/Marc/Documents/Uni/4t/TFG/Practices/Use Case 2/data/raw/pickle_2019-05-13-on7_2min.pkl
Please place 'pickle_2019-05-13-on7_2min.pkl' in /mnt/c/Users/Marc/Documents/Uni/4t/TFG/Practices/Use Case 2/data/raw/

In [ ]:
# Basic statistics
print("=== Target variable: Bytes ===")
print(raw_data["Bytes"].describe())
print(f"\nZero-load samples: {(raw_data['Bytes'] == 0).sum()} "
      f"({(raw_data['Bytes'] == 0).mean()*100:.1f}%)")
print(f"\n=== Active Users ===")
print(raw_data["Active Users"].describe())

## 2. Visualize AP Load Distributions
Reproducing Figure 2 from the paper — histograms + KDE for sample APs

In [ ]:
unique_aps = raw_data["AP ID"].unique()
sample_aps = np.random.choice(unique_aps, min(5, len(unique_aps)), replace=False)

fig, axes = plt.subplots(1, len(sample_aps), figsize=(4*len(sample_aps), 4))
if len(sample_aps) == 1:
    axes = [axes]

for ax, ap_id in zip(axes, sample_aps):
    ap_data = raw_data[raw_data["AP ID"] == ap_id]["Bytes"]
    # Log-transform for better visualization (as done in preprocessing)
    ap_log = np.log10(ap_data[ap_data > 0] * 1e-3)
    ax.hist(ap_log, bins=50, density=True, alpha=0.7, color="steelblue")
    ax.set_title(f"AP: {ap_id}")
    ax.set_xlabel("log10(KB)")
    ax.set_ylabel("Density")

plt.suptitle("Load Distribution per AP (log-scale)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Samples per AP
samples_per_ap = raw_data.groupby("AP ID").size()
print(f"Samples per AP — min: {samples_per_ap.min()}, max: {samples_per_ap.max()}, "
      f"median: {samples_per_ap.median():.0f}")

fig, ax = plt.subplots(figsize=(10, 4))
samples_per_ap.sort_values().plot(kind="bar", ax=ax, color="steelblue", alpha=0.7)
ax.set_xlabel("AP ID")
ax.set_ylabel("Number of samples")
ax.set_title("Samples per AP")
ax.set_xticks([])  # too many APs to label
plt.tight_layout()
plt.show()

## 3. Feature Engineering
The preprocessing pipeline (from `dataset.py`) does:
- Zero balancing: downsample zero-load rows to match non-zero count per AP
- Log transform: `Bytes → log10(Bytes * 1e-3)`
- Cyclical encoding: day_of_week and hour_of_day → sin/cos
- MinMax scaling to [1, 2]
- Time series windowing: lookback=60, steps={1,5,15,30}

**Features (6)**: Bytes, day_of_week_sin, day_of_week_cos, 
hour_of_day_sin, hour_of_day_cos, Active Users

In [ ]:
features = ["Bytes", "day_of_week_sin", "day_of_week_cos",
            "hour_of_day_sin", "hour_of_day_cos", "Active Users"]
print("Feature columns used for training:")
for f in features:
    print(f"  {f}: range [{raw_data[f].min():.4f}, {raw_data[f].max():.4f}]")

## 4. Generate Dirichlet Partitions

For each α ∈ {0.5, 1.0, 5.0, 10.0}, generate partitioned data
across 20 users (deployments). Lower α = more heterogeneity.

In [ ]:
ALPHAS = [0.5, 1.0, 5.0, 10.0]
LOOKBACK = 60
STEPS = 1

for alpha in ALPHAS:
    uc2.generate_partitions(alpha=alpha, lookback=LOOKBACK, steps=STEPS)

## 5. Verify Partitions & Visualize Heterogeneity

In [ ]:
import torch

for alpha in ALPHAS:
    part_path = os.path.join(
        uc2.DATA_PART, f"lookback_{LOOKBACK}", f"steps_{STEPS}",
        f"u20-alpha{alpha}-ratio1", "train", "train.pt"
    )
    with open(part_path, "rb") as f:
        data = torch.load(f, weights_only=False)

    samples = data["num_samples"]
    print(f"\nα={alpha}: {len(samples)} users, "
          f"total={sum(samples)} samples")
    print(f"  per-user: min={min(samples)}, max={max(samples)}, "
          f"std={np.std(samples):.1f}, CV={np.std(samples)/np.mean(samples):.2f}")

In [ ]:
# Visualize sample distribution across users for each alpha
fig, axes = plt.subplots(1, len(ALPHAS), figsize=(5*len(ALPHAS), 4), sharey=True)

for ax, alpha in zip(axes, ALPHAS):
    part_path = os.path.join(
        uc2.DATA_PART, f"lookback_{LOOKBACK}", f"steps_{STEPS}",
        f"u20-alpha{alpha}-ratio1", "train", "train.pt"
    )
    with open(part_path, "rb") as f:
        data = torch.load(f, weights_only=False)

    samples = data["num_samples"]
    ax.bar(range(len(samples)), sorted(samples, reverse=True),
           color="steelblue", alpha=0.7)
    ax.set_title(f"α = {alpha}")
    ax.set_xlabel("User (sorted)")
    if ax == axes[0]:
        ax.set_ylabel("# Training Samples")

plt.suptitle("Data Distribution Across Users (Dirichlet Partitioning)", fontsize=14)
plt.tight_layout()
plt.show()

## Summary

- Raw dataset loaded and explored: ~7400 APs, 49 days, 2-min windows
- Partitions generated for α ∈ {0.5, 1.0, 5.0, 10.0} with 20 users each
- Lower α → more heterogeneous distribution → harder FL problem
- Data saved in `data/partitions/lookback_60/steps_1/u20-alpha{α}-ratio1/`